In [ ]:
# notebook 07, would an LLM stand-in have known? Kernel: Python (yjmob-gpu), GPU session.
# Each test person becomes a persona built only from their ordinary six weeks (days 0-44). A language model, playing that person, is told
# that an emergency starts tomorrow and asked whether their routine will change. Its probability of 'yes' is scored against what happened:
# whose habit agent broke (notebook 04). Two stand-ins: Qwen3-8B, a general LLM, and Minitaur-8B, the small sibling of Centaur, a model
# fine-tuned on about ten million human choices from psychology experiments (Binz et al., Nature 2025)
import os, sys, json, time, zipfile
from pathlib import Path
os.environ["TORCH_DISABLE_NATIVE_JIT"] = "1"                      # plain PyTorch, as in notebook 05
ROOT = Path("/projects/ancz7294/yjmob")
os.environ.setdefault("HF_HOME", str(ROOT.parent / ".cache/huggingface"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.metrics import roc_auc_score
PROC, FIG, SAMPLE = ROOT / "data/processed", ROOT / "figures", ROOT / "data/samples"
SEED = 20260917
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
    "font.family": "sans-serif", "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight"})
def save(fig, name): fig.savefig(FIG / f"{name}.png"); plt.show(); plt.close(fig)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
ORDER = ["far roamers", "mid-range movers", "regular commuters", "local regulars", "weekend movers", "homebodies"]
FIFTH = ["least regular", "2nd", "3rd", "4th", "most regular"]
T = pd.read_parquet(PROC / "who_breaks_table.parquet")             # notebook 04: traits from days 0-44, the label, the split
te = T[T.split == "eval"].copy()                                    # the trees' test people, the same people every model is scored on
te["broke"] = te.broke.astype(bool)
te["fifth"] = pd.qcut(te.regularity, 5, labels=FIFTH)
risk_file = next(p for p in [PROC / "fm_tabpfn35_test_scores.parquet", PROC / "fm_tabicl_test_scores.parquet"] if p.exists())
te["tabular_fm"] = pd.read_parquet(risk_file).p.to_numpy()          # notebook 05: the pretrained tabular model, same order
N_PEOPLE = None                                                     # None = all test people; a number = a random subset
if N_PEOPLE: te = te.sample(N_PEOPLE, random_state=SEED)
print(f"{len(te):,} test people, {te.broke.mean():.1%} broke | tabular model: {risk_file.name}")

# the persona: the person's ordinary weeks in plain words, nothing from the emergency, no kind label, no place that could be located
def weekend(r):
    return "more often than" if r.weekend_away_ratio > 1.15 else ("less often than" if r.weekend_away_ratio < 0.85 else "about as often as")
def persona(r):
    place = (f"On {r.regularity:.0%} of workdays you spent the middle of the day at your usual daytime place, {r.commute_km:.1f} km from home."
             if r.commute_km >= 0.5 else f"Your usual daytime place is in your home neighbourhood; on {r.regularity:.0%} of workdays you spent the middle of the day there.")
    return "\n".join([
        f"- About {r.home_share:.0%} of the times your location was recorded, you were at home.",
        f"- On workdays you were away from home {r.morning_away:.0%} of the time between 7 and 9 am, {r.midday_away:.0%} between 10 am and 4 pm, "
        f"and {r.evening_away:.0%} between 6 and 10 pm.",
        f"- {place}",
        f"- You usually moved within about {r.rg_km:.1f} km and passed through about {r.cells_per_day:.0f} different 500-metre squares a day.",
        f"- On weekends you were away from home {weekend(r)} on workdays."])
QUESTION = ("Tomorrow an emergency begins in your city and will last about two weeks. You do not know yet what kind of emergency it is. "
            "Will your daily routine, meaning where you are at each time of the day, change noticeably compared with the last six weeks?")
INTRO = "Here is how you have lived over the last six weeks, from the location records of your phone:"
te["persona"] = [persona(r) for r in te.itertuples()]
print(te.persona.iloc[0])

In [ ]:
# the two stand-ins read every persona once; the model's next word after the question gives P(yes) against P(no). No text is generated,
# so this is one forward pass per person. Each model's scores are saved as soon as it finishes
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
MODELS = {
    "Qwen3-8B": dict(repo="Qwen/Qwen3-8B", chat=True, yes="Yes", no="No"),
    "Minitaur-8B": dict(repo="marcelbinz/Llama-3.1-Minitaur-8B", chat=False, yes="Y", no="N"),
}
def prompt_for(cfg, tok, p):
    if cfg["chat"]:                                                 # an instruction model: the question as a chat turn, thinking switched off
        msgs = [{"role": "user", "content": f"{INTRO}\n{p}\n\n{QUESTION}\nAnswer with Yes or No only."}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    # Minitaur and Centaur read experiments written as text, with the participant's choice between << and >>
    return (f"You will read how you have lived over the last six weeks, as recorded by your phone's location, and then answer a question "
            f"about the next two weeks.\n\n{p}\n\n{QUESTION} Press Y for yes or N for no.\nYou press <<")
@torch.no_grad()
def p_yes(name, cfg, texts, bs=32):
    tok = AutoTokenizer.from_pretrained(cfg["repo"]); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    y_id, n_id = tok.encode(cfg["yes"], add_special_tokens=False)[0], tok.encode(cfg["no"], add_special_tokens=False)[0]
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(cfg["repo"], dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32, device_map=DEVICE).eval()
    print(f"{name}: loaded in {time.time() - t0:.0f} s")
    prompts, out = [prompt_for(cfg, tok, p) for p in texts], []
    t0 = time.time()
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=not cfg["chat"]).to(DEVICE)
        try: logits = model(**enc, logits_to_keep=1).logits[:, -1, :].float()
        except TypeError: logits = model(**enc).logits[:, -1, :].float()
        out.append(torch.softmax(torch.stack([logits[:, y_id], logits[:, n_id]], 1), 1)[:, 0].cpu().numpy())
    secs = time.time() - t0
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"{name}: {len(prompts):,} people in {secs:.0f} s")
    return np.concatenate(out), secs, prompts[0]
LLM = {}
for name, cfg in MODELS.items():
    f_ = PROC / f"llm_{name.lower().replace('-', '_').replace('.', '_')}_scores.parquet"
    if f_.exists() and len(pd.read_parquet(f_)) == len(te):        # already done in an earlier run of this cell
        LLM[name] = pd.read_parquet(f_).p.to_numpy(); print(f"{name}: read from {f_.name}"); continue
    p_, secs, example = p_yes(name, cfg, te.persona.tolist())
    LLM[name] = p_
    pd.DataFrame({"uid": te.index, "p": p_}).to_parquet(f_)
    print("example prompt:\n" + example[-700:])
for name, p_ in LLM.items(): te[name] = p_

In [ ]:
# how good are the stand-ins? AUC for 'broke' over all test people, with a 95% interval from 2,000 bootstrap resamples, next to
# regularity alone and the pretrained tabular model of notebook 05; then the same within each fifth of regularity
scores = {"regularity alone": te.regularity.to_numpy(), "tabular model (notebook 05)": te.tabular_fm.to_numpy(),
          **{f"{n} stand-in": te[n].to_numpy() for n in LLM}}
y = te.broke.to_numpy()
idx = np.random.default_rng(SEED).integers(0, len(y), (2000, len(y)))
rows = []
for nm, s in scores.items():
    bs = np.array([roc_auc_score(y[i], s[i]) for i in idx])
    rows.append((nm, roc_auc_score(y, s), *np.percentile(bs, [2.5, 97.5])))
auc = pd.DataFrame(rows, columns=["score", "AUC", "2.5%", "97.5%"]).set_index("score")
print(auc.round(3).to_string())
by_fifth = te.groupby("fifth", observed=True).apply(lambda g: pd.Series({nm: roc_auc_score(g.broke, g[c]) for nm, c in
           [("tabular model", "tabular_fm")] + [(f"{n} stand-in", n) for n in LLM]}), include_groups=False)
print(by_fifth.round(3).to_string())

In [ ]:
# what do the stand-ins believe? Their average P(yes) by fifth of regularity and by kind, next to the share whose routine really broke.
# Rank correlation across the six kinds: +1 if a stand-in orders the kinds exactly as reality does, -1 if exactly the other way round
from scipy.stats import spearmanr
belief_f = te.groupby("fifth", observed=True)[["broke"] + list(LLM)].mean()
belief_k = te.groupby("kind")[["broke"] + list(LLM)].mean().reindex(ORDER)
print(belief_f.round(3).to_string()); print(belief_k.round(3).to_string())
rho = {n: spearmanr(belief_k.broke, belief_k[n]).statistic for n in LLM}
rho_f = {n: spearmanr(belief_f.broke, belief_f[n]).statistic for n in LLM}
print("rank correlation with reality, across kinds:", {k: round(v, 2) for k, v in rho.items()}, "| across fifths:", {k: round(v, 2) for k, v in rho_f.items()})
print("mean P(yes):", {n: round(float(te[n].mean()), 3) for n in LLM}, f"| real share broke {te.broke.mean():.3f}")
spread = {n: (round(float(belief_f[n].min()), 3), round(float(belief_f[n].max()), 3), round(float(te[n].std()), 3)) for n in LLM}
print("lowest and highest fifth, and spread between people (sd):", spread, f"| reality {belief_f.broke.min():.3f} to {belief_f.broke.max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1, 1.2]})
yy = np.arange(len(auc))[::-1]
col = ["#7d838c", C[0]] + [C[2], C[3]][: len(LLM)]
axes[0].errorbar(auc.AUC * 100, yy, xerr=[(auc.AUC - auc["2.5%"]) * 100, (auc["97.5%"] - auc.AUC) * 100], fmt="none", ecolor="#898781", lw=1.4, capsize=3)
axes[0].scatter(auc.AUC * 100, yy, color=col, s=60, zorder=3)
for v, y_ in zip(auc.AUC, yy): axes[0].text(v * 100, y_ + 0.22, f"{v * 100:.1f}", ha="center", fontsize=8.5, color="#52514e")
axes[0].axvline(50, color="#52514e", ls="--", lw=1); axes[0].set_yticks(yy); axes[0].set_yticklabels(auc.index); axes[0].yaxis.grid(False); axes[0].set_ylim(-0.6, len(auc) - 0.25)
axes[0].set(xlabel="AUC (%) for 'routine broke', all test people, 95% interval", title="Who will break? Stand-ins against the data")
xx = np.arange(5)
axes[1].bar(xx, belief_f.broke * 100, color="#f6c9b6", width=0.6, label="really broke")
for x_, v in zip(xx, belief_f.broke): axes[1].text(x_, v * 100 + 1.5, f"{v:.0%}", ha="center", fontsize=8.5, color="#52514e")
for n, c_ in zip(LLM, [C[2], C[3]]):
    axes[1].plot(xx, belief_f[n] * 100, color=c_, marker="o", lw=2, label=f"{n}: P(yes), average")
axes[1].set_xticks(xx); axes[1].set_xticklabels([f.replace(" ", "\n", 1) for f in FIFTH], fontsize=8.5); axes[1].xaxis.grid(False)
axes[1].set(ylabel="%", ylim=(0, 102), title="What the stand-ins expect, by how regular people were")
axes[1].legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3)
save(fig, "f52_llm_stand_ins")

In [ ]:
# the finding in plain words, a small sample of personas with the stand-ins' answers for the site, the log and the archive
best = auc.drop(["regularity alone", "tabular model (notebook 05)"]).AUC.idxmax()
print(f"""
Asked to play {len(te):,} real people from their ordinary six weeks alone, the stand-ins could not tell whose routine would break:
AUC {', '.join(f"{n} {auc.loc[f'{n} stand-in', 'AUC']:.3f}" for n in LLM)}, where regularity alone reaches {auc.loc['regularity alone', 'AUC']:.3f} and the tabular model
trained on the same weeks {auc.loc['tabular model (notebook 05)', 'AUC']:.3f}. They gave nearly everyone the same answer: {'; '.join(f"{n} said yes with {belief_f[n].min():.0%} to {belief_f[n].max():.0%} across the fifths of regularity" for n in LLM)},
while in reality {belief_f.broke.min():.0%} to {belief_f.broke.max():.0%} broke, {te.broke.mean():.0%} overall.
""")
ex = pd.concat([te[te.broke].sample(3, random_state=SEED), te[~te.broke].sample(3, random_state=SEED)])
ex[["persona", "broke"] + list(LLM)].round(3).to_csv(SAMPLE / "llm_personas_sample.csv")
log = {"people": len(te), "broke_share": round(float(te.broke.mean()), 4), "models": {n: MODELS[n]["repo"] for n in LLM}, "question": QUESTION,
       "auc": auc.round(4).to_dict(orient="index"), "auc_by_fifth": by_fifth.round(4).to_dict(orient="index"),
       "belief_by_fifth": belief_f.round(4).to_dict(orient="index"), "belief_by_kind": belief_k.round(4).to_dict(orient="index"),
       "rank_corr_kinds": {k: round(float(v), 3) for k, v in rho.items()}, "rank_corr_fifths": {k: round(float(v), 3) for k, v in rho_f.items()},
       "answer_spread": {n: {"lowest_fifth": v[0], "highest_fifth": v[1], "sd_between_people": v[2]} for n, v in spread.items()}}
(ROOT / "logs/part3_llm_log.json").write_text(json.dumps(log, indent=2, default=str))
out = ROOT / "part3_llm_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in [FIG / "f52_llm_stand_ins.png", ROOT / "logs/part3_llm_log.json", SAMPLE / "llm_personas_sample.csv", ROOT / "notebooks/07_llm_stand_ins.ipynb"]:
        if p.exists(): z.write(p, str(p.relative_to(ROOT)))
        else: print("missing:", p)
print(out, f"{out.stat().st_size / 1e6:.1f} MB (save the notebook before this cell so the archive holds it with its outputs)")